# Bank Customer Segmentation

This notebook performs customer segmentation using K-Means clustering. It covers selection of the optimal number of clusters with the elbow method, visualization of the resulting groups, profiling of customer segments, and generation of business insights.

## 1. Import Libraries

We import the libraries needed for clustering, visualization, and summary analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Dataset

The processed dataset is loaded for segmentation analysis. The notebook uses the scaled feature set prepared earlier.

In [2]:
processed_dir = Path("../datasets/processed")

# Load the processed feature matrix
X_scaled = pd.read_csv(processed_dir / "X_train_scaled.csv")
print("Loaded feature matrix shape:", X_scaled.shape)
X_scaled.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\datasets\\processed\\X_train_scaled.csv'

## 3. Standardize the Data

K-Means works best when features are on a comparable scale, so we use the already scaled data directly.

In [ ]:
# Use the scaled data as the clustering input
cluster_data = X_scaled.copy()
print("Clustering input shape:", cluster_data.shape)
cluster_data.head()

## 4. Determine the Optimal Number of Clusters with the Elbow Method

We evaluate a range of cluster counts and observe how the within-cluster sum of squares changes.

In [ ]:
# Calculate inertia for different values of k
inertia = []
cluster_range = range(1, 11)

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(cluster_data)
    inertia.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 5))
plt.plot(cluster_range, inertia, marker="o", linewidth=2, markersize=8)
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(cluster_range)
plt.grid(True)
plt.tight_layout()
plt.show()

# Suggest the elbow point as the preferred number of clusters
print("Inertia values by cluster count:")
for k, value in zip(cluster_range, inertia):
    print(f"k={k}: inertia={value:.2f}")

## 5. Fit K-Means Clustering

We train a K-Means model using the selected number of clusters.

In [ ]:
# Choose the number of clusters based on the elbow plot
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_data)

print("Optimal number of clusters selected:", optimal_k)
cluster_data["Cluster"] = clusters
cluster_data["Cluster"] = cluster_data["Cluster"].astype(int)
cluster_data.head()

## 6. Visualize the Clusters

We use PCA to visualize the clusters in two dimensions for easier interpretation.

In [ ]:
# Apply PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
pca_components = pca.fit_transform(cluster_data.drop(columns=["Cluster"]))

pca_df = pd.DataFrame(pca_components, columns=["PC1", "PC2"])
pca_df["Cluster"] = clusters

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cluster", palette="Set2", s=80)
plt.title("Customer Segments Visualized with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 7. Profile Customer Segments

We profile each cluster by summarizing its average feature values.

In [ ]:
# Add cluster labels back to the original scaled features
segmentation_df = X_scaled.copy()
segmentation_df["Cluster"] = clusters

# Profile each cluster
cluster_profile = segmentation_df.groupby("Cluster").mean().round(3)
print("Cluster profile summary:")
print(cluster_profile)

## 8. Business Insights

We translate the clustering results into practical business recommendations.

In [ ]:
print("Business insights:")
print("- Cluster 0 may represent customers with moderate engagement and average churn risk.")
print("- Cluster 1 may represent a high-risk or high-value segment that should be monitored closely.")
print("- Cluster 2 may represent low-risk customers with stable behavior.")
print("- Cluster 3 may represent a distinct group that may require tailored retention actions.")
print("- These segments can guide targeted marketing, loyalty offers, and churn prevention strategies.")